# Experiment no 05

## 1. Aim
To classify observations from a suitable multivariate open-access dataset using regularized linear models and study how L2 and L1 regularization influence generalization, model complexity, and multicollinearity.

## 2. Objectives
1. Load and inspect a multivariate classification dataset.
2. Preprocess the data using a train–test split and feature standardization.
3. Build a Ridge classifier (L2 regularization) and an L1-regularized logistic classifier (Lasso-style regularization).
4. Compare training and test performance using accuracy, precision, recall, F1-score, and confusion matrices.
5. Observe coefficient shrinkage and feature selection.
6. Examine multicollinearity through feature correlations and the variance inflation factor (VIF) for a selected subset of features.
7. Draw conclusions about regularization and overfitting.

## 3. Theory
Regularization adds a penalty to a model's objective function to discourage unnecessarily large coefficients. It can reduce variance and improve performance on unseen observations.
- **Ridge / L2 penalty:** penalizes the sum of squared coefficients. Coefficients are reduced toward zero, but rarely become exactly zero.
- **Lasso / L1 penalty:** penalizes the sum of absolute coefficients. It can set some coefficients exactly to zero, performing a form of feature selection.
- **Classification note:** Ordinary Ridge and Lasso are regression estimators. For classification, this experiment uses `RidgeClassifier` for L2 regularization and `LogisticRegression(penalty='l1')` for Lasso-style regularization.
- The regularization strength is controlled by `alpha` for `RidgeClassifier`. For logistic regression, `C` is the inverse of regularization strength: smaller `C` means stronger regularization.

## 4. Dataset
**Wisconsin Diagnostic Breast Cancer dataset**, available through Scikit-learn (`load_breast_cancer`).
- 569 observations, 30 numeric input features.
- Binary target: malignant (0) or benign (1).


### Step 1 — Import required libraries


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Libraries imported successfully.")


### Step 2 — Load and inspect the dataset


In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Dataset shape:", X.shape)
print("Target classes:", dict(enumerate(dataset.target_names)))
print("\nClass distribution:")
print(y.map(dict(enumerate(dataset.target_names))).value_counts())

display(X.head())
display(X.describe().T.head(10))


#### Target encoding and distribution check


In [ ]:
print("Missing values in features:", int(X.isna().sum().sum()))
print("Missing target values:", int(y.isna().sum()))

plt.figure(figsize=(6, 4))
y.map(dict(enumerate(dataset.target_names))).value_counts().plot(kind="bar")
plt.title("Class distribution")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Step 3 — Explore multicollinearity


In [ ]:
# Correlation heatmap for a selected set of related measurements
selected_features = [
    "mean radius", "mean perimeter", "mean area",
    "worst radius", "worst perimeter", "worst area"
]

plt.figure(figsize=(9, 6))
sns.heatmap(X[selected_features].corr(), annot=True, cmap="coolwarm",
            vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation heatmap of selected features")
plt.tight_layout()
plt.show()


#### Variance Inflation Factor (VIF)


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X[selected_features].copy()
X_vif_scaled = StandardScaler().fit_transform(X_vif)

vif_table = pd.DataFrame({
    "Feature": selected_features,
    "VIF": [
        variance_inflation_factor(X_vif_scaled, i)
        for i in range(X_vif_scaled.shape[1])
    ]
}).sort_values("VIF", ascending=False)

display(vif_table)
print("Rule of thumb: VIF values above 5 (or 10 in a stricter convention) may indicate substantial multicollinearity.")


### Step 4 — Split data and standardize features


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Training class counts:")
print(y_train.value_counts())


### Step 5 — Define the models


In [ ]:
models = {
    "Logistic baseline (weak L2)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1e6, max_iter=10000, random_state=RANDOM_STATE))
    ]),
    "Ridge Classifier (L2)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier(alpha=1.0))
    ]),
    "L1 Logistic (Lasso-style)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1", solver="liblinear", C=1.0,
            max_iter=10000, random_state=RANDOM_STATE
        ))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")


### Step 6 — Evaluate training and test performance


In [ ]:
results = []

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    results.append({
        "Model": name,
        "Train accuracy": accuracy_score(y_train, train_pred),
        "Test accuracy": accuracy_score(y_test, test_pred),
        "Test precision": precision_score(y_test, test_pred, zero_division=0),
        "Test recall": recall_score(y_test, test_pred, zero_division=0),
        "Test F1": f1_score(y_test, test_pred, zero_division=0),
        "Train-test accuracy gap": (
            accuracy_score(y_train, train_pred) - accuracy_score(y_test, test_pred)
        )
    })

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))


### Step 7 — Confusion matrices and classification reports


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=dataset.target_names,
        cmap="Blues", ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name, model in models.items():
    print("\n" + "=" * 75)
    print(name)
    print(classification_report(
        y_test, model.predict(X_test),
        target_names=dataset.target_names, zero_division=0
    ))


### Step 8 — Cross-validation on the training set


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    cv_rows.append({
        "Model": name,
        "CV accuracy mean": scores.mean(),
        "CV accuracy std": scores.std(),
        "Fold scores": np.round(scores, 4).tolist()
    })

cv_df = pd.DataFrame(cv_rows).set_index("Model")
display(cv_df)


### Step 9 — Compare coefficient shrinkage and feature selection


In [ ]:
# RidgeClassifier coefficients
ridge_est = models["Ridge Classifier (L2)"].named_steps["model"]
ridge_coef = ridge_est.coef_.ravel()

# LogisticRegression coefficients
baseline_est = models["Logistic baseline (weak L2)"].named_steps["model"]
baseline_coef = baseline_est.coef_.ravel()

l1_est = models["L1 Logistic (Lasso-style)"].named_steps["model"]
l1_coef = l1_est.coef_.ravel()

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Weak-L2 logistic": baseline_coef,
    "Ridge classifier": ridge_coef,
    "L1 logistic": l1_coef
})

display(coef_df.head(15).round(4))

summary_coef = pd.DataFrame({
    "Model": ["Weak-L2 logistic", "Ridge classifier", "L1 logistic"],
    "Number of non-zero coefficients": [
        int(np.count_nonzero(np.abs(baseline_coef) > 1e-8)),
        int(np.count_nonzero(np.abs(ridge_coef) > 1e-8)),
        int(np.count_nonzero(np.abs(l1_coef) > 1e-8))
    ],
    "L2 norm of coefficients": [
        float(np.linalg.norm(baseline_coef)),
        float(np.linalg.norm(ridge_coef)),
        float(np.linalg.norm(l1_coef))
    ]
})
display(summary_coef.round(4))


#### Feature coefficients bar comparison


In [ ]:
top_features = coef_df.reindex(
    coef_df["L1 logistic"].abs().sort_values(ascending=False).index
).head(12)

plot_df = top_features.set_index("Feature")[[
    "Weak-L2 logistic", "Ridge classifier", "L1 logistic"
]]

plot_df.plot(kind="bar", figsize=(13, 5))
plt.title("Coefficient comparison for features with largest L1 coefficients")
plt.ylabel("Coefficient (standardized inputs)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


### Step 10 — Study regularization strength


In [ ]:
strength_rows = []

for alpha in [0.01, 0.1, 1, 10, 100]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier(alpha=alpha))
    ])
    model.fit(X_train, y_train)
    strength_rows.append({
        "Model": "Ridge",
        "Strength parameter": alpha,
        "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
        "Test accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Non-zero coefficients": int(np.count_nonzero(
            np.abs(model.named_steps["model"].coef_) > 1e-8
        )),
        "Coefficient L2 norm": float(np.linalg.norm(model.named_steps["model"].coef_))
    })

for C in [0.01, 0.1, 1, 10, 100]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1", solver="liblinear", C=C,
            max_iter=10000, random_state=RANDOM_STATE
        ))
    ])
    model.fit(X_train, y_train)
    strength_rows.append({
        "Model": "L1 logistic",
        "Strength parameter": C,
        "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
        "Test accuracy": accuracy_score(y_test, model.predict(X_test)),
        "Non-zero coefficients": int(np.count_nonzero(
            np.abs(model.named_steps["model"].coef_) > 1e-8
        )),
        "Coefficient L2 norm": float(np.linalg.norm(model.named_steps["model"].coef_))
    })

strength_df = pd.DataFrame(strength_rows)
display(strength_df.round(4))


#### Plot effect of regularization strength


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ridge_strength = strength_df[strength_df["Model"] == "Ridge"]
axes[0].plot(ridge_strength["Strength parameter"], ridge_strength["Train accuracy"], marker="o", label="Train")
axes[0].plot(ridge_strength["Strength parameter"], ridge_strength["Test accuracy"], marker="s", label="Test")
axes[0].set_xscale("log")
axes[0].set_title("Ridge: effect of alpha")
axes[0].set_xlabel("alpha (larger = stronger regularization)")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

l1_strength = strength_df[strength_df["Model"] == "L1 logistic"]
axes[1].plot(l1_strength["Strength parameter"], l1_strength["Train accuracy"], marker="o", label="Train")
axes[1].plot(l1_strength["Strength parameter"], l1_strength["Test accuracy"], marker="s", label="Test")
axes[1].set_xscale("log")
axes[1].set_title("L1 logistic: effect of C")
axes[1].set_xlabel("C (smaller = stronger regularization)")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 16. Results and observations
- **Best test performance:** Compare the model with the highest test F1-score and accuracy.
- **Overfitting:** The weakly regularized model often shows an accuracy gap between train and test. Regularization narrows this gap.
- **Multicollinearity:** Features like mean radius, mean perimeter, and mean area exhibit severe multicollinearity (VIF > 100).
- **Ridge effect:** Shrinks coefficient magnitudes evenly across correlated predictors without setting them to zero.
- **L1 effect:** Performs sparse feature selection, setting irrelevant or redundant feature coefficients to zero.

## 17. Conclusion
Ridge and L1 regularization control model complexity by penalizing large coefficients. Ridge (L2) typically distributes shrinkage across correlated predictors, stabilizing estimates. L1-regularized logistic regression sets some coefficients to zero, providing a simpler model and implicit feature selection.

## 18. Viva voce questions and Answers

**Q1. What is regularization in machine learning?**  
*Answer:* Regularization is a technique that adds a penalty term to the loss function to penalize extreme model complexity, prevent overfitting, and improve generalization.

**Q2. What is the difference between L1 and L2 penalties?**  
*Answer:* L2 (Ridge) penalizes the sum of squared weights ($\lambda \sum w_i^2$), shrinking weights toward zero; L1 (Lasso) penalizes the sum of absolute weights ($\lambda \sum |w_i|$), driving weights to exact zeros and producing sparse models.

**Q3. Why should features be standardized before applying regularized linear models?**  
*Answer:* Penalties treat all coefficient magnitudes equally. Unstandardized features with large numerical units would have small coefficients and artificially escape penalization.

**Q4. What is multicollinearity, and how can it affect coefficient estimates?**  
*Answer:* Multicollinearity occurs when predictor variables are highly correlated. It leads to unstable coefficient estimates with high variance.

**Q5. What does a high VIF indicate?**  
*Answer:* Variance Inflation Factor (VIF) measures how much the variance of an estimated regression coefficient increases due to collinearity. A VIF > 5 or 10 indicates severe multicollinearity.

**Q6. Why does L1 regularization produce sparse coefficients?**  
*Answer:* Geometrically, the diamond-shaped L1 constraint has sharp corners on the coordinate axes where the loss function contours frequently touch first, forcing coefficients to exactly 0.

**Q7. Why is C inverse to regularization strength in logistic regression?**  
*Answer:* In Scikit-learn, the objective is formulated as $C \times \text{Loss} + \text{Penalty}$. Therefore, smaller $C$ places relatively greater weight on the penalty (stronger regularization).

**Q8. Why must preprocessing be fitted only on training data?**  
*Answer:* Fitting transformers on test data causes data leakage, as test statistics leak into the model and produce overly optimistic performance evaluations.

**Q9. Why is cross-validation useful when selecting a regularization parameter?**  
*Answer:* It provides an unbiased estimate of generalization performance across multiple training folds without touching the held-out test set.

**Q10. Why should a test set not be repeatedly used to tune hyperparameters?**  
*Answer:* Repeated tuning against the test set leaks test information into the selection process, causing the test set to lose its objectivity as a measure of true generalization error.
